# Reconstruction vs contrastive vs prediction

The pre-registered design, the hypotheses and their thresholds are in `experiments/recon_vs_pred/README.md`, which was written before any learner was trained.

**Question.** Does a prediction objective keep a dynamical regime that a reconstruction objective loses? If so, is the reason that it drops what it cannot predict?

**Manipulation.** One regime-independent nuisance is added to the observation. Its timescale is either fast (weak start-to-start correlation; residual predictability across the gap) or slow (persists into the target), at amplitude σ_n = 1 or 2, plus a no-nuisance condition N0.

This notebook only *displays* results that were computed elsewhere:
- `python experiments/recon_vs_pred/run.py pilot` runs the local, exploratory pilot;
- `python experiments/recon_vs_pred/run.py hpc <runs dir>` scores the confirmatory HPC sweep.

Set `MODE` to choose which results are shown.

In [ ]:
import os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, "experiments/recon_vs_pred")
import cpbench
from cpbench.data import Dataset
from design import CONDITIONS, dataset, W
from experiment import verdicts, PRED, RECON
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "font.size": 9})
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 120)

MODE = os.environ.get("CPBENCH_MODE", "pilot")      # ✏️ "pilot" (exploratory) or "hpc" (confirmatory)
df = pd.read_csv(f"cpbench_results/recon_vs_pred_{MODE}.csv")
print(MODE, len(df), "rows;", sorted(df.cond.unique()), "seeds", sorted(df.seed.unique()))

## 1. The data

Each panel shows the first 400 samples of three of the 64 channels in one condition. The coloured band marks the regime. The nuisance is invisible in N0, wiggles quickly in F2 and drifts slowly in S2.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 2.6), sharey=True)
for ax, c in zip(axes, ("N0", "F2", "S2")):
    d = Dataset(dataset(c, quick=True)).sample(1, 1200, np.random.default_rng(0))
    for j in range(3):
        ax.plot(d["obs"][0, :400, j] + 3 * j, lw=0.7)
    ax.pcolorfast((0, 400), ax.get_ylim(), d["S_s"][:, :400], cmap="Pastel1", alpha=0.35)
    ax.set_title(f"{c}: nuisance {CONDITIONS[c][0]}, sigma_n = {CONDITIONS[c][1]}"); ax.set_xlabel("sample")
plt.show()

## 2. Verdicts (pre-registered thresholds)

In pilot mode these verdicts are exploratory: one seed and 3k steps. Only the HPC run decides H1–H4.

In [ ]:
verdicts(df, confirmatory=MODE == "hpc")

## 3. All numbers (mean ± sd over seeds)

| Column | What it measures | Reference |
|---|---|---|
| `probe` | linear probe of the regime | chance 0.5 |
| `probe_mlp` | MLP probe on window pairs, secondary | — |
| `nuis_r2` | linear decoding of nuisance coordinates | 0 = test-mean baseline; negative values retained |
| `hmm` | unsupervised clustering accuracy, primary | — |
| `gmm` | unsupervised clustering accuracy, secondary | — |

In [ ]:
cols = ["probe", "probe_mlp", "nuis_r2", "hmm", "gmm"]
order = ["ae", "vae", "jepa_separate", "jepa_shared", "cpc", "ts2vec", "untrained", "raw_flat", "raw_lagcov", "oracle"]
g = df.groupby(["cond", "learner"])[cols]
tab = g.mean().round(3).astype(str) + np.where(g.count() > 1, " ± " + g.std().round(2).astype(str), "")
tab = tab.reset_index().pivot(index="learner", columns="cond").reindex(order).dropna(how="all")
tab

## 4. What each learner keeps

Each point is one run. The x-axis shows linear nuisance decodability, and the y-axis how much of the regime (linear probe).

Hypothesis H2 predicts:
- **prediction** (JEPA) sits to the left in F2, if the fast nuisance becomes less linearly accessible, and moves right in S2;
- **reconstruction** (AE, VAE) sits to the right in both.

In [ ]:
COL = {"reconstruction": "tab:red", "prediction": "tab:blue", "contrastive (future target)": "tab:cyan",
       "contrastive (same-time target)": "tab:orange", "control": "0.6"}
conds = [c for c in CONDITIONS if c in set(df.cond)]
fig, axes = plt.subplots(1, len(conds), figsize=(3.3 * len(conds), 3.2), sharey=True, sharex=True)
for ax, c in zip(np.atleast_1d(axes), conds):
    for fam, sub in df[df.cond == c].groupby("family"):
        ax.scatter(sub.nuis_r2, sub.probe, color=COL[fam], s=28, label=fam, alpha=0.85, edgecolor="k", lw=0.3)
        for r in sub.drop_duplicates("learner").itertuples():
            ax.annotate(r.learner, (r.nuis_r2, r.probe), fontsize=6, xytext=(3, 2), textcoords="offset points")
    ax.axhline(0.5, color="0.8", lw=0.8); ax.set_title(c); ax.set_xlabel("nuisance: linear decoding R²")
np.atleast_1d(axes)[0].set_ylabel("regime: linear probe"); np.atleast_1d(axes)[0].legend(fontsize=6, loc="lower left")
plt.tight_layout(); plt.show()

## 5. Retention versus clusterability

Each panel compares the regime probe (bars) with unsupervised HMM accuracy (black dots). A high probe with low HMM accuracy indicates information accessible to the probe but poorly recovered by this detector; it does not prove that every clustering method fails.

In [ ]:
lead = [l for l in order if l in set(df.learner)]
fig, axes = plt.subplots(1, len(conds), figsize=(3.3 * len(conds), 3.0), sharey=True)
for ax, c in zip(np.atleast_1d(axes), conds):
    m = df[df.cond == c].groupby("learner")[["probe", "hmm"]].mean().reindex(lead)
    x = np.arange(len(m))
    ax.bar(x, m.probe, color=[COL[df[df.learner == l].family.iloc[0]] for l in m.index], alpha=0.6)
    ax.scatter(x, m.hmm, color="k", s=14, zorder=3)
    ax.set_xticks(x, m.index, rotation=70, fontsize=7); ax.axhline(0.5, color="0.7", lw=0.8); ax.set_title(c)
np.atleast_1d(axes)[0].set_ylabel("bars: probe   dots: HMM acc")
plt.tight_layout(); plt.show()

## 6. Dose response (needs all five conditions: the HPC run)

Each metric is plotted against the nuisance amplitude σ_n (0, 1, 2). Solid lines are fast nuisance and dashed lines slow nuisance.

In [ ]:
if set(CONDITIONS) <= set(df.cond):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
    for ax, m in zip(axes, ("probe", "nuis_r2", "hmm")):
        for l in ["ae", "vae", "jepa_separate", "jepa_shared", "cpc", "ts2vec"]:
            fam = df[df.learner == l].family.iloc[0]
            for speed, ls in (("fast", "-"), ("slow", "--")):
                sub = df[(df.learner == l) & ((df.speed == speed) | (df.cond == "N0"))].groupby("sigma_n")[m].mean()
                ax.plot(sub.index, sub.values, ls, color=COL[fam], marker="o", ms=3, label=l if speed == "fast" else None)
        ax.set_xlabel("nuisance sigma_n"); ax.set_title(m)
    axes[0].legend(fontsize=6); plt.tight_layout(); plt.show()
else:
    print("pilot: only", conds)

## Audit amendment
The primary thresholds are unchanged. Confirmatory verdicts require complete audit_v2 results and passing gates. Three paired training seeds on one fixed dataset seed quantify training variability only. TS2Vec uses its reference architecture; this is not an objective-only, equal-capacity comparison.
CP metrics and optional nonlinear/paired nuisance diagnostics are secondary. HMM/GMM primary fits are selected by persistence. See the dated amendment in experiments/recon_vs_pred/README.md.


In [ ]:
secondary = [c for c in df.columns if "_cp_" in c or c in ("nuis_mlp_r2", "nuis_paired_distance", "hmm_fixed_acc", "gmm_fixed_acc", "parameters_total")]
if secondary:
    display(df.groupby(["cond", "learner"])[secondary].agg(["mean", "std"]))
